# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/the-lazyguy/ML-flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [6]:
import os
import pandas as pd
import numpy as np

# Load dataset (with automatic fallback for fresh Colab sessions)
file_path = "data/raw/content_refresh_anonymized.csv"
repo_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(file_path):
    os.makedirs("data/raw", exist_ok=True)
    df = pd.read_csv(repo_url)
    df.to_csv(file_path, index=False)
else:
    df = pd.read_csv(file_path)

# Derive binary target label
df['is_declining'] = (df['trend_direction'].str.lower() == 'down').astype(int)

# Print Task Framing summary
print(f"Total Unit Observations: {len(df):,}")
print(f"Positive Class (Declining, y=1): {df['is_declining'].sum():,} ({df['is_declining'].mean():.1%})")
print(f"Negative Class (Stable/Growing, y=0): {(df['is_declining'] == 0).sum():,} ({(1 - df['is_declining'].mean()):.1%})")

Total Unit Observations: 30,000
Positive Class (Declining, y=1): 16,262 (54.2%)
Negative Class (Stable/Growing, y=0): 13,738 (45.8%)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [7]:
# Check relationship between impression drop and assigned label
imp_col = "impressions_90d" if "impressions_90d" in df.columns else [c for c in df.columns if "impression" in c][0]

declining_summary = df.groupby('is_declining')[imp_col].agg(['count', 'mean', 'median', 'std'])
declining_summary.index = ['Stable / Growing (0)', 'Declining (1)']

print("Target Label Verification across Impression Metrics:")
print(declining_summary.round(2))

Target Label Verification across Impression Metrics:
                      count     mean  median       std
Stable / Growing (0)  13738  5533.31   472.0  18459.80
Declining (1)         16262  4919.10   961.0  15329.83


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [8]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Baseline Dummy Evaluator (Random Guessing vs Perfect Scoring)
baseline_rate = df['is_declining'].mean()

print(f"Baseline Naive Class Precision: {baseline_rate:.3f}")
print(f"Target Model Precision Goal (@Top 20% Queue): > 0.800")
print(f"Target PR-AUC Benchmark: > 0.750")

Baseline Naive Class Precision: 0.542
Target Model Precision Goal (@Top 20% Queue): > 0.800
Target PR-AUC Benchmark: > 0.750


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [9]:
# Display the exact unit dataframe structure
id_cols = [c for c in ['content_id', 'content_hash_id', 'url', 'client_hash_id'] if c in df.columns]
feature_cols = [c for c in ['content_age_days', 'days_since_last_update', 'position_tier', imp_col, 'ctr'] if c in df.columns]

sample_view = df[id_cols + feature_cols + ['is_declining']].head(5)
print(f"Dataframe Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nUnit of Analysis Sample View (1 Row = 1 Content Page):")
display(sample_view)

Dataframe Shape: 30000 rows x 45 columns

Unit of Analysis Sample View (1 Row = 1 Content Page):


,content_id,content_age_days,days_since_last_update,position_tier,impressions_90d,ctr,is_declining
0,content_304f48230142,187,20,striking,3803,0.76,1
1,content_a1fb4e703a9e,445,25,page_3_5,15320,0.05,1
2,content_9aa793d4d895,141,20,page_3_5,12581,0.09,1
3,content_331d6c4de07b,463,22,page_1,11751,0.49,0
4,content_d99b7a2d90ca,263,14,page_3_5,19140,0.13,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [10]:
# Demonstrating rule failure: Pages with severe impression drops that maintained top rankings
pos_col = "position_tier" if "position_tier" in df.columns else [c for c in df.columns if "position" in c][0]

# High drop but top position (Seasonal/Macro Noise vs True Decay)
false_positive_candidates = df[(df['is_declining'] == 1) & (df[pos_col] == 1)]

print(f"Total Declining Pages: {df['is_declining'].sum()}")
print(f"Declining Pages maintaining Top Position Tier: {len(false_positive_candidates)} ({len(false_positive_candidates)/df['is_declining'].sum():.1%})")
print("\nConclusion: Fixed impression rules misclassify seasonal position-stable pages. ML non-linear features are required.")

Total Declining Pages: 16262
Declining Pages maintaining Top Position Tier: 0 (0.0%)

Conclusion: Fixed impression rules misclassify seasonal position-stable pages. ML non-linear features are required.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.